In [9]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# 1. Load dataset
df = pd.read_csv("garments_worker_productivity.csv")


# 2. Create classification target
df["MeetsTarget"] = (
    df["actual_productivity"]
    >= df["targeted_productivity"]
).astype(int)


# 3. Select features
features = [
    "quarter",
    "department",
    "day",
    "team",
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]


# 4. X and targets
X = df[features]

y_regression = df["actual_productivity"]
y_classification = df["MeetsTarget"]


# 5. Fixed train-test split
(
    X_train,
    X_test,
    y_reg_train,
    y_reg_test,
    y_cls_train,
    y_cls_test
) = train_test_split(
    X,
    y_regression,
    y_classification,
    test_size=0.20,
    random_state=42
)


# 6. Define feature types
categorical_features = [
    "quarter",
    "department",
    "day",
    "team"
]

numerical_features = [
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]


# 7. Numerical preprocessing
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


# 8. Categorical preprocessing
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])


# 9. Combine preprocessing
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numerical_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])


# 10. Preprocess
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)


# ==========================================================
# LINEAR REGRESSION
# ==========================================================

linear_model = LinearRegression()


start = time.perf_counter()

linear_model.fit(
    X_train_processed,
    y_reg_train
)

regression_training_time = (
    time.perf_counter() - start
)


start = time.perf_counter()

reg_predictions = linear_model.predict(
    X_test_processed
)

regression_prediction_time = (
    time.perf_counter() - start
)


mae = mean_absolute_error(
    y_reg_test,
    reg_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_reg_test,
        reg_predictions
    )
)

r2 = r2_score(
    y_reg_test,
    reg_predictions
)


# ==========================================================
# LOGISTIC REGRESSION
# ==========================================================

logistic_model = LogisticRegression(
    max_iter=500,
    random_state=42
)


start = time.perf_counter()

logistic_model.fit(
    X_train_processed,
    y_cls_train
)

classification_training_time = (
    time.perf_counter() - start
)


start = time.perf_counter()

cls_predictions = logistic_model.predict(
    X_test_processed
)

classification_prediction_time = (
    time.perf_counter() - start
)


accuracy = accuracy_score(
    y_cls_test,
    cls_predictions
)

precision = precision_score(
    y_cls_test,
    cls_predictions,
    zero_division=0
)

recall = recall_score(
    y_cls_test,
    cls_predictions,
    zero_division=0
)

f1 = f1_score(
    y_cls_test,
    cls_predictions,
    zero_division=0
)


# ==========================================================
# RESULTS
# ==========================================================

print("\n===== LINEAR REGRESSION =====")

print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R2   : {r2:.6f}")

print(
    f"Training time   : "
    f"{regression_training_time:.6f} seconds"
)

print(
    f"Prediction time : "
    f"{regression_prediction_time:.6f} seconds"
)


print("\n===== LOGISTIC REGRESSION =====")

print(f"Accuracy  : {accuracy:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1        : {f1:.6f}")

print(
    f"Training time   : "
    f"{classification_training_time:.6f} seconds"
)

print(
    f"Prediction time : "
    f"{classification_prediction_time:.6f} seconds"
)


===== LINEAR REGRESSION =====
MAE  : 0.112110
RMSE : 0.150504
R2   : 0.146915
Training time   : 0.005897 seconds
Prediction time : 0.000327 seconds

===== LOGISTIC REGRESSION =====
Accuracy  : 0.762500
Precision : 0.791262
Recall    : 0.920904
F1        : 0.851175
Training time   : 0.021043 seconds
Prediction time : 0.000368 seconds


In [1]:
import pandas as pd
import numpy as np
import time


# ==========================================================
# 1. LOAD DATASET
# ==========================================================

df = pd.read_csv("garments_worker_productivity.csv")


# ==========================================================
# 2. CREATE CLASSIFICATION TARGET
# ==========================================================

df["MeetsTarget"] = (
    df["actual_productivity"]
    >= df["targeted_productivity"]
).astype(int)


# ==========================================================
# 3. SELECT FEATURES
# ==========================================================

features = [
    "quarter",
    "department",
    "day",
    "team",
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]


# ==========================================================
# 4. X AND TARGETS
# ==========================================================

X = df[features].copy()

y_regression = df["actual_productivity"].values
y_classification = df["MeetsTarget"].values


# ==========================================================
# 5. FIXED TRAIN-TEST SPLIT
#    Equivalent to an 80/20 split with random_state=42
# ==========================================================

np.random.seed(42)

indices = np.random.permutation(len(X))

test_size = int(0.20 * len(X))

test_indices = indices[:test_size]
train_indices = indices[test_size:]

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()

y_reg_train = y_regression[train_indices]
y_reg_test = y_regression[test_indices]

y_cls_train = y_classification[train_indices]
y_cls_test = y_classification[test_indices]


# ==========================================================
# 6. FEATURE TYPES
# ==========================================================

categorical_features = [
    "quarter",
    "department",
    "day",
    "team"
]

numerical_features = [
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]


# ==========================================================
# 7. NUMERICAL PREPROCESSING
#    Median imputation + Standardization
# ==========================================================

# Store preprocessing parameters from TRAINING data only

numeric_medians = {}
numeric_means = {}
numeric_stds = {}

for column in numerical_features:

    # Median calculated only from training data
    median_value = X_train[column].median()

    numeric_medians[column] = median_value

    X_train[column] = X_train[column].fillna(
        median_value
    )

    X_test[column] = X_test[column].fillna(
        median_value
    )

    # Mean and standard deviation from training data
    mean_value = X_train[column].mean()
    std_value = X_train[column].std()

    # Avoid division by zero
    if std_value == 0:
        std_value = 1.0

    numeric_means[column] = mean_value
    numeric_stds[column] = std_value

    # Standardize
    X_train[column] = (
        X_train[column] - mean_value
    ) / std_value

    X_test[column] = (
        X_test[column] - mean_value
    ) / std_value


# ==========================================================
# 8. CATEGORICAL PREPROCESSING
#    Most-frequent imputation + One-hot encoding
# ==========================================================

categorical_categories = {}

for column in categorical_features:

    # Find most frequent value from training data
    mode = X_train[column].mode()

    if len(mode) > 0:
        most_frequent = mode.iloc[0]
    else:
        most_frequent = "Unknown"

    # Fill missing values
    X_train[column] = X_train[column].fillna(
        most_frequent
    )

    X_test[column] = X_test[column].fillna(
        most_frequent
    )

    # Categories are learned ONLY from training data
    categories = sorted(
        X_train[column].astype(str).unique()
    )

    categorical_categories[column] = categories


# ==========================================================
# 9. MANUAL ONE-HOT ENCODING
# ==========================================================

train_parts = []
test_parts = []


# Numerical columns
train_numeric = X_train[
    numerical_features
].values.astype(float)

test_numeric = X_test[
    numerical_features
].values.astype(float)

train_parts.append(train_numeric)
test_parts.append(test_numeric)


# Categorical columns
for column in categorical_features:

    categories = categorical_categories[column]

    train_values = X_train[column].astype(str).values
    test_values = X_test[column].astype(str).values

    train_encoded = np.zeros(
        (len(X_train), len(categories))
    )

    test_encoded = np.zeros(
        (len(X_test), len(categories))
    )

    for j, category in enumerate(categories):

        train_encoded[:, j] = (
            train_values == category
        ).astype(float)

        test_encoded[:, j] = (
            test_values == category
        ).astype(float)

    train_parts.append(train_encoded)
    test_parts.append(test_encoded)


# Combine all features
X_train_processed = np.hstack(train_parts)
X_test_processed = np.hstack(test_parts)


# ==========================================================
# ADD INTERCEPT COLUMN
# ==========================================================

X_train_with_bias = np.column_stack([
    np.ones(len(X_train_processed)),
    X_train_processed
])

X_test_with_bias = np.column_stack([
    np.ones(len(X_test_processed)),
    X_test_processed
])


# ==========================================================
# LINEAR REGRESSION
# ==========================================================

start = time.perf_counter()


# Normal Equation:
#
# beta = (X^T X)^-1 X^T y
#
# pinv is used instead of inverse because it is
# numerically more stable.

linear_coefficients = np.linalg.pinv(
    X_train_with_bias
) @ y_reg_train


regression_training_time = (
    time.perf_counter() - start
)


# ==========================================================
# LINEAR REGRESSION PREDICTION
# ==========================================================

start = time.perf_counter()

reg_predictions = (
    X_test_with_bias
    @ linear_coefficients
)

regression_prediction_time = (
    time.perf_counter() - start
)


# ==========================================================
# REGRESSION METRICS
# ==========================================================

errors = (
    y_reg_test
    - reg_predictions
)

mae = np.mean(
    np.abs(errors)
)

rmse = np.sqrt(
    np.mean(errors ** 2)
)

ss_res = np.sum(
    errors ** 2
)

ss_tot = np.sum(
    (y_reg_test - np.mean(y_reg_test)) ** 2
)

r2 = 1 - (
    ss_res / ss_tot
)


# ==========================================================
# LOGISTIC REGRESSION
# ==========================================================

def sigmoid(z):

    # Prevent numerical overflow
    z = np.clip(z, -500, 500)

    return 1 / (
        1 + np.exp(-z)
    )


# Initialize weights
weights = np.zeros(
    X_train_with_bias.shape[1]
)


# Hyperparameters
learning_rate = 0.01
epochs = 5000


start = time.perf_counter()


# Gradient Descent
for epoch in range(epochs):

    # Linear combination
    z = (
        X_train_with_bias
        @ weights
    )

    # Sigmoid probability
    probabilities = sigmoid(z)

    # Gradient
    gradient = (
        X_train_with_bias.T
        @ (
            probabilities
            - y_cls_train
        )
    ) / len(y_cls_train)

    # Update weights
    weights -= (
        learning_rate
        * gradient
    )


classification_training_time = (
    time.perf_counter() - start
)


# ==========================================================
# LOGISTIC REGRESSION PREDICTION
# ==========================================================

start = time.perf_counter()

test_probabilities = sigmoid(
    X_test_with_bias @ weights
)

cls_predictions = (
    test_probabilities >= 0.5
).astype(int)


classification_prediction_time = (
    time.perf_counter() - start
)


# ==========================================================
# CLASSIFICATION METRICS
# ==========================================================

true_positive = np.sum(
    (y_cls_test == 1)
    & (cls_predictions == 1)
)

true_negative = np.sum(
    (y_cls_test == 0)
    & (cls_predictions == 0)
)

false_positive = np.sum(
    (y_cls_test == 0)
    & (cls_predictions == 1)
)

false_negative = np.sum(
    (y_cls_test == 1)
    & (cls_predictions == 0)
)


# Accuracy
accuracy = (
    true_positive + true_negative
) / len(y_cls_test)


# Precision
if (
    true_positive + false_positive
) == 0:

    precision = 0.0

else:

    precision = (
        true_positive
        / (
            true_positive
            + false_positive
        )
    )


# Recall
if (
    true_positive + false_negative
) == 0:

    recall = 0.0

else:

    recall = (
        true_positive
        / (
            true_positive
            + false_negative
        )
    )


# F1 Score
if (
    precision + recall
) == 0:

    f1 = 0.0

else:

    f1 = (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )


# ==========================================================
# RESULTS
# ==========================================================

print("\n===== LINEAR REGRESSION — WITHOUT SCIKIT-LEARN =====")

print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R2   : {r2:.6f}")

print(
    f"Training time   : "
    f"{regression_training_time:.6f} seconds"
)

print(
    f"Prediction time : "
    f"{regression_prediction_time:.6f} seconds"
)


print("\n===== LOGISTIC REGRESSION — WITHOUT SCIKIT-LEARN =====")

print(f"Accuracy  : {accuracy:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1        : {f1:.6f}")

print(
    f"Training time   : "
    f"{classification_training_time:.6f} seconds"
)

print(
    f"Prediction time : "
    f"{classification_prediction_time:.6f} seconds"
)


===== LINEAR REGRESSION — WITHOUT SCIKIT-LEARN =====
MAE  : 0.112457
RMSE : 0.150819
R2   : 0.146514
Training time   : 0.269232 seconds
Prediction time : 0.000222 seconds

===== LOGISTIC REGRESSION — WITHOUT SCIKIT-LEARN =====
Accuracy  : 0.761506
Precision : 0.769231
Recall    : 0.965909
F1        : 0.856423
Training time   : 1.286748 seconds
Prediction time : 0.000455 seconds
